# Phase 1 cycle pretrain — revision 4 (embedding-only, no LLM)

**Spec** (Stage 1, Colab Pro feasible):
- Encoder: `BAAI/bge-large-en-v1.5` (1024d, 334M frozen, public — no HF_TOKEN needed)
- MoE: K_routed=16 + K_shared=4, d_hidden=2048, sparsegen + global λ (use_user=False)
- FactDecoder: tight 64d bottleneck
- Total trainable: ~62M

**Corpus** (revision 4 Option B — multi-source for broader cognitive context):
- Pennebaker Essays (~2.5k, 1 essay/user, Big Five labelled)
- Reddit 8 subreddits (programming / politics / cooking / science / relationships / personalfinance / AskHistorians / philosophy, ~80k rows scan/subreddit, active user ≥5 comments, **cross-subreddit author dedup** — same user across subreddits = single user_id)
- PANDORA (jingjietan/pandora-big5, Reddit users × OCEAN, ≥5 comments/user)
- PersonaChat (bavard/personachat_truecased, persona-grounded dialogue responses)
- ≈ 150-200k samples total. Cycle pretrain ↔ plan §3.1 evaluation domain alignment (Reddit/PANDORA/PersonaChat/Pennebaker) — no out-of-domain gap.
- **First run**: ~60-90 min for HF stream (8 subreddits × 80k rows + PANDORA + PersonaChat) + BGE encode (~200k samples). Cached at `out/phase1/cache/` — subsequent runs skip.
- Per training run: ~15h on L4 24GB (Pro).

**This notebook runs**:
1. λ_ortho = 0.0 (paradigm minimal — emergent specialization via cycle alone)
2. λ_ortho = 0.05 (SIMoE orthogonality reg, explicit specialization pressure)
3. Eval both — cycle reconstruction quality + Nikolic 2025 linear probe separability

If Stage 1 results promising → escalate to Stage 2 (Expanded #4, K=64/16, ~500M, Pro+/cloud).

See `phase1/README.md` for full architecture + Stage 2 spec.

## 1. Setup — Drive mount + path

In [ ]:
from google.colab import drive
import os, sys
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

# Drive 구조: 내 드라이브 / sideproject / phase1
# BASE = phase1 의 *parent* (sys.path 에 추가해서 `from phase1.train import` 동작)
BASE = '/content/drive/MyDrive/sideproject'

# 한국어 Colab 환경은 `MyDrive` 가 `내 드라이브` 폴더로 mount 될 수도 — fallback.
if not os.path.isdir(BASE):
    alt = '/content/drive/내 드라이브/sideproject'
    if os.path.isdir(alt):
        BASE = alt
    else:
        raise FileNotFoundError(
            f'BASE not found: tried {BASE} and {alt}\n'
            f"확인: print(os.listdir('/content/drive'))"
        )
if not os.path.isdir(os.path.join(BASE, 'phase1')):
    raise FileNotFoundError(f'phase1/ 폴더가 {BASE} 안에 없음 — Drive sync 확인')

os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)

# Persist HuggingFace caches on Drive so re-runs skip re-download.
hf_cache = Path(BASE) / '.hf_cache'
hf_cache.mkdir(exist_ok=True)
os.environ.setdefault('HF_HOME', str(hf_cache))
os.environ.setdefault('HF_DATASETS_CACHE', str(hf_cache / 'datasets'))

print('BASE     =', BASE)
print('HF_HOME  =', os.environ['HF_HOME'])
print('cwd      =', os.getcwd())

## 2. Install dependencies

BGE-large-en is public — no HF_TOKEN required.

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = ['datasets', 'transformers', 'accelerate', 'sentencepiece', 'sacrebleu', 'tqdm']
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print('installing:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
else:
    print('all required packages already present')

import torch
print('torch =', torch.__version__, ' cuda =', torch.cuda.is_available(),
      ' device =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Sanity test (CPU, ~10s)

15 unit tests to confirm the codebase imports cleanly before any GPU work.

In [ ]:
!python -m pytest phase1/tests/test_phase1_ablation.py -v

## 4. (Optional) Pre-build corpus + BGE encoding cache

Runs the multi-source corpus assembly + BGE-large encode once **before** the training cells, so the two training runs share the same cache and the first training cell's startup is cheap.

Cost: ~10-30 min for first run (HF stream of 4 sources + 200k × BGE encode). Cached at `out/phase1/cache/corpus_<hash>.parquet` + `fact_emb_*.npy`. Skip this cell if you already have cache from a previous session.

**If you want a smaller corpus** (faster sanity test): edit `cfg = CorpusConfig(include_pandora=False, include_personachat=False, reddit_splits=('programming',))` and re-run.

**The training cells below will auto-build if you skip this**, but then *Stage 1 Run 1* will absorb the build cost.

In [ ]:
from phase1.data import CorpusConfig, load_phase1_corpus

cfg = CorpusConfig()   # default = Option B (Pennebaker + 8 subreddits + PANDORA + PersonaChat)
corpus, fact_emb = load_phase1_corpus(cfg=cfg)
print(f'corpus: {len(corpus)} samples, {corpus["user_id"].nunique()} users')
print(f'fact_emb: {fact_emb.shape}')
print('source counts:')
print(corpus['source'].value_counts())

## 5. Stage 1 — Run 1: λ_ortho = 0.0

Paradigm-minimal — emergent specialization via cycle reconstruction alone (no explicit orthogonality pressure).

First ~300 steps run as the inline **sanity gate** (NaN/Inf, OOM, K_active>0, loss-decreasing checks). If sanity passes, full 10-epoch training continues automatically.

**Per-epoch checkpoint** (encoder weights stripped, ~250 MB):
- `model.pt` — trainable weights only (eval 용)
- `ckpt.pt` — full state (model + optimizer + epoch + RNG + history + sanity verdict)

**Auto-resume across Colab session timeouts**: 12h Pro session timeout 안에 학습 ~15h 다 못 끝남 → session 끊기면 *notebook 다시 열어 이 cell 재실행*. `train_phase1` 가 `out/phase1/<run_id>/ckpt.pt` 자동 detect → optimizer state + RNG + epoch counter 모두 복원, 다음 epoch 부터 이어 학습. Clean restart 가 필요하면 `out/phase1/ph1_v0_ortho_off/` 폴더 삭제 후 재실행.

In [ ]:
from phase1.train import train_phase1

train_phase1(
    run_id='ph1_v0_ortho_off',
    epochs=10,
    batch_size=32,
    lr=1e-4,
    lambda_ortho=0.0,
    sanity_steps=300,
)

## 6. Stage 1 — Run 2: λ_ortho = 0.05

SIMoE orthogonality reg on — explicit pressure for expert disentanglement. Same hyperparameters otherwise (corpus cache shared from Run 1).

In [ ]:
from phase1.train import train_phase1

train_phase1(
    run_id='ph1_v0_ortho_on',
    epochs=10,
    batch_size=32,
    lr=1e-4,
    lambda_ortho=0.05,
    sanity_steps=300,
)

## 7. Eval — cycle reconstruction + Nikolic 2025 linear probe

Two metrics on the held-out test set (10% sample-level holdout):
- **Cycle reconstruction**: mean cosine + MSE of `fact_emb` vs `fact_emb_recon`
- **Linear probe separability**: SVM on `routed_alpha` and `sub_kg` → source label (Pennebaker / Reddit / PANDORA / PersonaChat), as coarse cognitive-context proxy

In [ ]:
from phase1.eval import eval_phase1

result_off = eval_phase1(run_id='ph1_v0_ortho_off')
print('\n' + '=' * 60)
result_on  = eval_phase1(run_id='ph1_v0_ortho_on')

## 8. Result review + Stage 2 decision

Compare cycle reconstruction quality + probe accuracy across the two runs. Reference: Nikolic 2025 = *unsupervised expert > supervised by +8.3pp on QuickDraw*.

**Stage 2 escalation criteria** (RESEARCH_PLAN §3.4 revision 4):
- Phase 1 > B0 (Generic Encoder + CapacityMLP + FactDecoder) on cycle BLEU/cosine
- Phase 1 > B1 (Generic Encoder + Standard MoE + FactDecoder) on cycle BLEU/cosine
- λ_ortho on > λ_ortho off in linear probe (validates SIMoE pressure adds value)

Promising → request Pro+ or cloud GPU, swap to Expanded #4 (K_routed=64, K_shared=16, d_hidden=3072, ~500M trainable). **Then** write `eval_simbench_classifier.py` (Path A) + `eval_simbench_llm.py` (Path B) for SimBench external eval (1등 candidate).

**B0/B1 baseline runs are not in this notebook** — they need a separate train script (`phase1/baselines/train_baselines.py`, future).

In [ ]:
import json
from pathlib import Path

for run_id in ('ph1_v0_ortho_off', 'ph1_v0_ortho_on'):
    p = Path('out/phase1') / run_id / 'eval.json'
    if not p.exists():
        print(f'{run_id}: eval.json not found, skipping')
        continue
    r = json.loads(p.read_text())
    recon = r['cycle_reconstruction']
    pa = r['linear_probe_routed_alpha']
    pk = r['linear_probe_sub_kg']
    print(f'\n=== {run_id} ===')
    print(f'  cycle_recon  mean_cosine={recon["mean_cosine"]:.4f}  mse={recon["mean_mse"]:.6f}')
    print(f'  probe alpha  acc={pa.get("accuracy", "n/a"):.4f}  macro_f1={pa.get("macro_f1", "n/a"):.4f}')
    print(f'  probe subkg  acc={pk.get("accuracy", "n/a"):.4f}  macro_f1={pk.get("macro_f1", "n/a"):.4f}')